In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import RFECV
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier

data = pd.read_csv('Graph metrics.csv') 

X = data.drop(['Class', 'SUBID'], axis=1)
y = data['Class']

feature_mapping = {feature: idx for idx, feature in enumerate(X.columns)}


X.columns = [feature_mapping[feature] for feature in X.columns]


feature_mapping_df = pd.DataFrame(list(feature_mapping.items()), columns=['Feature Name', 'Numerical Value'])
feature_mapping_df.to_excel('feature_mapping.xlsx', index=False)


X.to_csv('modified_X.csv', index=False)

scaler = MinMaxScaler()
X_normalized = scaler.fit_transform(X)  
X=X_normalized

features_arr = [5, 10, 20, 40, 50, 75, 100, 125, 150, 175, 200, 225, 250, 275, 300, 325, 350]

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

# Outer 5-fold cross-validation
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Store results
train_results = []
test_results = []

for num_features in features_arr:
    print(f"Testing with top {num_features} features")
    
    # Outer loop
    all_test_metrics = []
    all_train_metrics = []
    for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # Inner cross-validation for feature selection using Random Forest
        rf = RandomForestClassifier(n_estimators=100, random_state=42)
        rfecv = RFECV(estimator=rf, step=1, cv=5, n_jobs=-1, verbose=False)
        rfecv.fit(X_train, y_train)
        
        # Select top 'num_features' based on importance
        ranking = np.argsort(rfecv.ranking_)[:num_features]
        X_train_selected = X_train[:, ranking]
        X_test_selected = X_test[:, ranking]
        selected_features = list(ranking)
        selected_features_str = ", ".join(map(str, selected_features))

        # Perform GridSearchCV
        grid_search = GridSearchCV(
            RandomForestClassifier(random_state=42),
            param_grid, cv=5, n_jobs=-1, scoring='roc_auc'
        )
        grid_search.fit(X_train_selected, y_train)
        best_model = grid_search.best_estimator_
        best_params = grid_search.best_params_

        # Predict on test set
        y_test_pred = best_model.predict(X_test_selected)
        y_test_pred_proba = best_model.predict_proba(X_test_selected)[:, 1]
        auc_test = roc_auc_score(y_test, y_test_pred_proba)

        # Calculate test metrics
        tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred).ravel()
        accuracy_test = accuracy_score(y_test, y_test_pred)
        precision_test = precision_score(y_test, y_test_pred, average='weighted')
        recall_test = recall_score(y_test, y_test_pred, average='weighted')
        f1_test = f1_score(y_test, y_test_pred, average='weighted')
        specificity_test = tn / (tn + fp)
        
        # Predict on train set
        y_train_pred = best_model.predict(X_train_selected)
        y_train_pred_proba = best_model.predict_proba(X_train_selected)[:, 1]
        auc_train = roc_auc_score(y_train, y_train_pred_proba)

        # Calculate train metrics
        tn, fp, fn, tp = confusion_matrix(y_train, y_train_pred).ravel()
        accuracy_train = accuracy_score(y_train, y_train_pred)
        precision_train = precision_score(y_train, y_train_pred, average='weighted')
        recall_train = recall_score(y_train, y_train_pred, average='weighted')
        f1_train = f1_score(y_train, y_train_pred, average='weighted')
        specificity_train = tn / (tn + fp)

        
        test_results.append([
            num_features, selected_features_str, fold + 1,
            accuracy_test, precision_test, recall_test, 
            f1_test, auc_test, specificity_test, best_params
        ])

        
        train_results.append([
            num_features, selected_features_str, fold + 1,
            accuracy_train, precision_train, recall_train, 
            f1_train, auc_train, specificity_train, best_params
        ])

       
        all_test_metrics.append([
            accuracy_test, precision_test, recall_test, 
            f1_test, auc_test, specificity_test
        ])

        all_train_metrics.append([
            accuracy_train, precision_train, recall_train, 
            f1_train, auc_train, specificity_train
        ])

    
    avg_test_metrics = np.mean(all_test_metrics, axis=0)
    avg_train_metrics = np.mean(all_train_metrics, axis=0)

    
    test_results.append([
        num_features, selected_features_str, 'Average',
        *avg_test_metrics, best_params
    ])

    train_results.append([
        num_features, selected_features_str, 'Average',
        *avg_train_metrics, best_params
    ])


columns = [
    'Top Features', 'Selected Features', 'Fold', 
    'Accuracy', 'Precision', 'Recall', 'F1 Score', 
    'AUC', 'Specificity', 'Best Parameters'
]

train_df = pd.DataFrame(train_results, columns=columns)
test_df = pd.DataFrame(test_results, columns=columns)


train_df.to_excel('train_results_RF.xlsx', index=False)
test_df.to_excel('test_results_RF.xlsx', index=False)